<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта: Реализация класса Invoice</h2>

----

### Вариант задания №10


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Invoice в C#, который будет представлять информацию о фактурах за поставленные товары или оказанные услуги. На основе этого класса разработать 2-3 производных класса, демонстрирующих принципы наследования и полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и методы, а также переопределены некоторые методы базового класса для
демонстрации полиморфизма.
Требования к базовому классу Invoice:

• Атрибуты: Номер фактуры (InvoiceNumber), Дата выдачи (IssueDate), Общая
сумма (TotalAmount).

• Методы:

o CalculateTotal(): метод для расчета общей суммы по фактуре.

o AddLine(LineItem lineItem): метод для добавления позиции в фактуру.

o RemoveLine(LineItem lineItem): метод для удаления позиции из фактуры.

Требования к производным классам:

1. ТоварнаяФактура (GoodsInvoice): Должна содержать дополнительные атрибуты, такие как Дата поставки (SupplyDate). Метод AddLine() должен быть переопределен для добавления информации о дате поставки товара при добавлении позиции.

2. УслуговаяФактура (ServiceInvoice): Должна содержать дополнительные атрибуты, такие как Дата оказания услуги (ServiceDate). Метод RemoveLine() должен быть переопределен для добавления информации о причине аннулирования услуги при удалении позиции.

3. КомбинированнаяФактура (CombinedInvoice) (если требуется третий класс): Должна содержать дополнительные атрибуты, такие как Наличие возврата (ReturnAllowed). Метод CalculateTotal() должен быть переопределен для учета возможного возврата товара или услуги при расчете общей суммы.

#### Дополнительное задание
Добавьте к сущестующим классам конструктора классов с использованием гетторов и сетторов и реализуйте взаимодействие объектов между собой

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

// Базовый класс для позиции в фактуре
public class LineItem
{
    // Private поля (инкапсуляция)
    private string _description;
    private decimal _quantity;
    private decimal _unitPrice;

    // Свойства с геттерами и сеттерами + валидация
    public string Description
    {
        get { return _description; }
        set 
        { 
            if (!string.IsNullOrWhiteSpace(value) && value.Length <= 100)
                _description = value;
            else
                throw new ArgumentException("Некорректное описание позиции");
        }
    }

    public decimal Quantity
    {
        get { return _quantity; }
        set 
        { 
            if (value > 0)
                _quantity = value;
            else
                throw new ArgumentException("Количество должно быть больше 0");
        }
    }

    public decimal UnitPrice
    {
        get { return _unitPrice; }
        set 
        { 
            if (value >= 0)
                _unitPrice = value;
            else
                throw new ArgumentException("Цена не может быть отрицательной");
        }
    }

    // Public свойство только для чтения (вычисляемое)
    public decimal Total => Quantity * UnitPrice;

    // Конструктор с использованием свойств (геттеров/сеттеров)
    public LineItem(string description, decimal quantity, decimal unitPrice)
    {
        // Используем сеттеры для валидации данных
        Description = description;
        Quantity = quantity;
        UnitPrice = unitPrice;
    }

    // Конструктор по умолчанию
    public LineItem()
    {
        Description = "Без названия";
        Quantity = 1;
        UnitPrice = 0;
    }

    // Метод для клонирования позиции
    public LineItem Clone()
    {
        return new LineItem(Description, Quantity, UnitPrice);
    }

    public virtual decimal GetTotal()
    {
        return Total;
    }

    // Взаимодействие: сравнение с другой позицией
    public bool IsMoreExpensiveThan(LineItem other)
    {
        if (other == null) return true;
        return this.Total > other.Total;
    }

    // Взаимодействие: объединение с другой позицией (если одинаковое описание)
    public bool TryMergeWith(LineItem other)
    {
        if (other != null && this.Description.Equals(other.Description, StringComparison.OrdinalIgnoreCase))
        {
            this.Quantity += other.Quantity;
            // Пересчитываем среднюю цену, если цены разные
            if (this.UnitPrice != other.UnitPrice)
            {
                this.UnitPrice = (this.Total + other.Total) / this.Quantity;
            }
            return true;
        }
        return false;
    }

    public override string ToString()
    {
        return $"{Description} | Кол-во: {Quantity} | Цена: {UnitPrice:C} | Сумма: {Total:C}";
    }
}

// Базовый класс Invoice (Фактура)
public class Invoice
{
    // Private поля
    private string _invoiceNumber;
    private DateTime _issueDate;
    private decimal _totalAmount;
    protected List<LineItem> lineItems;

    // Свойства с геттерами и сеттерами
    public string InvoiceNumber
    {
        get { return _invoiceNumber; }
        protected set 
        { 
            if (!string.IsNullOrWhiteSpace(value) && value.Length <= 20)
                _invoiceNumber = value;
            else
                throw new ArgumentException("Некорректный номер фактуры");
        }
    }

    public DateTime IssueDate
    {
        get { return _issueDate; }
        protected set 
        { 
            if (value <= DateTime.Now)
                _issueDate = value;
            else
                throw new ArgumentException("Дата не может быть в будущем");
        }
    }

    public decimal TotalAmount
    {
        get { return _totalAmount; }
        protected set 
        { 
            if (value >= 0)
                _totalAmount = value;
        }
    }

    // Public свойство только для чтения для доступа к позициям
    public IReadOnlyList<LineItem> LineItems => lineItems.AsReadOnly();

    // Конструктор с использованием свойств (геттеров/сеттеров)
    public Invoice(string invoiceNumber, DateTime issueDate)
    {
        // Используем сеттеры для валидации
        InvoiceNumber = invoiceNumber;
        IssueDate = issueDate;
        lineItems = new List<LineItem>();
        TotalAmount = 0;
    }

    // Конструктор по умолчанию
    public Invoice() : this("UNKNOWN", DateTime.Now) { }

    // Методы
    public virtual void CalculateTotal()
    {
        TotalAmount = 0;
        foreach (var item in lineItems)
        {
            TotalAmount += item.GetTotal();
        }
        Console.WriteLine($"[Invoice {InvoiceNumber}] Общая сумма: {TotalAmount:C}");
    }

    public virtual void AddLine(LineItem lineItem)
    {
        if (lineItem != null)
        {
            // Проверяем, есть ли уже такая позиция - если да, объединяем
            var existingItem = lineItems.FirstOrDefault(
                x => x.Description.Equals(lineItem.Description, StringComparison.OrdinalIgnoreCase));
            
            if (existingItem != null)
            {
                existingItem.TryMergeWith(lineItem);
                Console.WriteLine($"[Invoice {InvoiceNumber}] Позиция '{lineItem.Description}' обновлена");
            }
            else
            {
                lineItems.Add(lineItem.Clone());
                Console.WriteLine($"[Invoice {InvoiceNumber}] Добавлена позиция: {lineItem.Description}");
            }
        }
    }

    public virtual void RemoveLine(LineItem lineItem)
    {
        if (lineItem != null)
        {
            var itemToRemove = lineItems.FirstOrDefault(
                x => x.Description.Equals(lineItem.Description, StringComparison.OrdinalIgnoreCase));
            
            if (itemToRemove != null)
            {
                lineItems.Remove(itemToRemove);
                Console.WriteLine($"[Invoice {InvoiceNumber}] Удалена позиция: {lineItem.Description}");
            }
        }
    }

    public void DisplayInfo()
    {
        Console.WriteLine($"\n=== Фактура №{InvoiceNumber} ===");
        Console.WriteLine($"Дата выдачи: {IssueDate.ToShortDateString()}");
        Console.WriteLine($"Количество позиций: {lineItems.Count}");
        Console.WriteLine($"Общая сумма: {TotalAmount:C}");
    }

    // === ВЗАИМОДЕЙСТВИЕ МЕЖДУ ОБЪЕКТАМИ ===

    // Взаимодействие: передача позиции из одной фактуры в другую
    public void TransferLineTo(Invoice targetInvoice, string description)
    {
        var item = lineItems.FirstOrDefault(
            x => x.Description.Equals(description, StringComparison.OrdinalIgnoreCase));
        
        if (item != null && targetInvoice != null)
        {
            lineItems.Remove(item);
            targetInvoice.AddLine(item);
            Console.WriteLine($"[TRANSFER] Позиция '{description}' перенесена из {InvoiceNumber} в {targetInvoice.InvoiceNumber}");
        }
    }

    // Взаимодействие: объединение двух фактур
    public void MergeWith(Invoice otherInvoice)
    {
        if (otherInvoice != null && otherInvoice != this)
        {
            Console.WriteLine($"\n[MERGE] Объединение фактур: {InvoiceNumber} + {otherInvoice.InvoiceNumber}");
            
            foreach (var item in otherInvoice.lineItems)
            {
                this.AddLine(item);
            }
            this.CalculateTotal();
            
            // Очищаем объединенную фактуру
            otherInvoice.lineItems.Clear();
            otherInvoice.TotalAmount = 0;
            Console.WriteLine($"[MERGE] Фактура {otherInvoice.InvoiceNumber} обнулена");
        }
    }

    // Взаимодействие: сравнение сумм двух фактур
    public bool IsGreaterThan(Invoice other)
    {
        if (other == null) return true;
        return this.TotalAmount > other.TotalAmount;
    }

    // Взаимодействие: получение общей суммы нескольких фактур
    public static decimal GetCombinedTotal(params Invoice[] invoices)
    {
        decimal total = 0;
        foreach (var inv in invoices)
        {
            if (inv != null)
                total += inv.TotalAmount;
        }
        return total;
    }
}

// Производный класс 1: Товарная фактура (GoodsInvoice)
public class GoodsInvoice : Invoice
{
    // Private поле
    private DateTime _supplyDate;

    // Свойство с геттером и сеттером
    public DateTime SupplyDate
    {
        get { return _supplyDate; }
        private set 
        { 
            if (value >= IssueDate) // Дата поставки не раньше даты фактуры
                _supplyDate = value;
            else
                throw new ArgumentException("Дата поставки не может быть раньше даты фактуры");
        }
    }

    // Конструктор с использованием свойств (геттеров/сеттеров)
    public GoodsInvoice(string invoiceNumber, DateTime issueDate, DateTime supplyDate) 
        : base(invoiceNumber, issueDate)
    {
        // Используем сеттер для валидации
        SupplyDate = supplyDate;
    }

    // Переопределение метода AddLine
    public override void AddLine(LineItem lineItem)
    {
        base.AddLine(lineItem);
        Console.WriteLine($"[GoodsInvoice] Товар '{lineItem.Description}' добавлен с датой поставки: {SupplyDate.ToShortDateString()}");
    }

    // Взаимодействие: проверка возможности поставки товара
    public bool CanSupplyTo(GoodsInvoice otherInvoice)
    {
        if (otherInvoice == null) return false;
        // Можно объединить поставки, если даты близки (в пределах 7 дней)
        return Math.Abs((SupplyDate - otherInvoice.SupplyDate).Days) <= 7;
    }

    // Взаимодействие: создание объединенной поставки
    public GoodsInvoice CreateCombinedSupply(GoodsInvoice other, string newInvoiceNumber)
    {
        if (CanSupplyTo(other))
        {
            var combined = new GoodsInvoice(
                newInvoiceNumber, 
                DateTime.Now, 
                SupplyDate > other.SupplyDate ? SupplyDate : other.SupplyDate);
            
            // Копируем позиции из обеих фактур
            foreach (var item in this.LineItems)
                combined.AddLine(item.Clone());
            foreach (var item in other.LineItems)
                combined.AddLine(item.Clone());
            
            combined.CalculateTotal();
            Console.WriteLine($"[GoodsInvoice] Создана объединенная поставка: {newInvoiceNumber}");
            return combined;
        }
        Console.WriteLine("[GoodsInvoice] Невозможно объединить поставки (даты слишком разные)");
        return null;
    }

    public void DisplaySupplyInfo()
    {
        Console.WriteLine($"Дата поставки: {SupplyDate.ToShortDateString()}");
    }
}

// Производный класс 2: Услуговая фактура (ServiceInvoice)
public class ServiceInvoice : Invoice
{
    // Private поле
    private DateTime _serviceDate;

    // Свойство с геттером и сеттером
    public DateTime ServiceDate
    {
        get { return _serviceDate; }
        private set 
        { 
            if (value >= IssueDate.AddDays(-30)) // Услуга не раньше чем за 30 дней до фактуры
                _serviceDate = value;
            else
                throw new ArgumentException("Дата услуги слишком давняя");
        }
    }

    // Конструктор с использованием свойств
    public ServiceInvoice(string invoiceNumber, DateTime issueDate, DateTime serviceDate) 
        : base(invoiceNumber, issueDate)
    {
        ServiceDate = serviceDate;
    }

    // Переопределение метода RemoveLine
    public override void RemoveLine(LineItem lineItem)
    {
        if (lineItem != null)
        {
            Console.WriteLine($"[ServiceInvoice] Аннулирование услуги '{lineItem.Description}' от {ServiceDate.ToShortDateString()}");
            Console.WriteLine($"- Причина: по требованию клиента или невыполнение условий");
            base.RemoveLine(lineItem);
        }
    }

    // Взаимодействие: проверка возможности отмены услуги
    public bool CanCancelService(ServiceInvoice other)
    {
        if (other == null) return false;
        // Можно отменить, если услуги одного типа и даты близки
        return this.LineItems.Any(i => i.Description.Contains("Услуга")) &&
               other.LineItems.Any(i => i.Description.Contains("Услуга"));
    }

    // Взаимодействие: перенос услуги в другую фактуру
    public void TransferServiceTo(ServiceInvoice target, string serviceName)
    {
        if (CanCancelService(target))
        {
            this.TransferLineTo(target, serviceName);
            Console.WriteLine($"[ServiceInvoice] Услуга '{serviceName}' перенесена в фактуру {target.InvoiceNumber}");
        }
    }

    public void DisplayServiceInfo()
    {
        Console.WriteLine($"Дата оказания услуги: {ServiceDate.ToShortDateString()}");
    }
}

// Производный класс 3: Комбинированная фактура (CombinedInvoice)
public class CombinedInvoice : Invoice
{
    // Private поля
    private bool _returnAllowed;
    private decimal _returnAmount;

    // Свойства с геттерами и сеттерами
    public bool ReturnAllowed
    {
        get { return _returnAllowed; }
        private set { _returnAllowed = value; }
    }

    public decimal ReturnAmount
    {
        get { return _returnAmount; }
        private set 
        { 
            if (value >= 0 && value <= TotalAmount)
                _returnAmount = value;
        }
    }

    // Конструктор с использованием свойств
    public CombinedInvoice(string invoiceNumber, DateTime issueDate, bool returnAllowed) 
        : base(invoiceNumber, issueDate)
    {
        ReturnAllowed = returnAllowed;
        ReturnAmount = 0;
    }

    // Переопределение метода CalculateTotal
    public override void CalculateTotal()
    {
        base.CalculateTotal();
        
        if (ReturnAllowed && ReturnAmount > 0)
        {
            decimal finalAmount = TotalAmount - ReturnAmount;
            Console.WriteLine($"[CombinedInvoice] Учтен возврат на сумму: {ReturnAmount:C}");
            Console.WriteLine($"[CombinedInvoice] Итоговая сумма: {finalAmount:C}");
            TotalAmount = finalAmount;
        }
        else if (ReturnAllowed)
        {
            Console.WriteLine($"[CombinedInvoice] Возврат разрешен, но возвратов не было");
        }
    }

    // Метод для добавления возврата
    public void AddReturn(decimal amount)
    {
        if (ReturnAllowed && amount >= 0)
        {
            ReturnAmount += amount;
            Console.WriteLine($"[CombinedInvoice] Добавлен возврат: {amount:C}");
            CalculateTotal(); // Пересчитываем итог
        }
        else if (!ReturnAllowed)
        {
            Console.WriteLine("[CombinedInvoice] Возврат не разрешен для данной фактуры!");
        }
    }

    // === ВЗАИМОДЕЙСТВИЕ ===

    // Взаимодействие: применение возврата из другой фактуры
    public void ApplyReturnFrom(CombinedInvoice other)
    {
        if (ReturnAllowed && other != null && other.ReturnAmount > 0)
        {
            Console.WriteLine($"[CombinedInvoice] Применяем возврат из фактуры {other.InvoiceNumber}");
            this.AddReturn(other.ReturnAmount);
        }
    }

    // Взаимодействие: объединение с товарной фактурой
    public void ImportFromGoodsInvoice(GoodsInvoice goodsInv)
    {
        if (goodsInv != null)
        {
            Console.WriteLine($"[CombinedInvoice] Импорт товаров из {goodsInv.InvoiceNumber}");
            foreach (var item in goodsInv.LineItems)
            {
                this.AddLine(item.Clone());
            }
            this.CalculateTotal();
        }
    }

    // Взаимодействие: объединение с услуговой фактурой
    public void ImportFromServiceInvoice(ServiceInvoice serviceInv)
    {
        if (serviceInv != null)
        {
            Console.WriteLine($"[CombinedInvoice] Импорт услуг из {serviceInv.InvoiceNumber}");
            foreach (var item in serviceInv.LineItems)
            {
                this.AddLine(item.Clone());
            }
            this.CalculateTotal();
        }
    }

    public void DisplayReturnInfo()
    {
        Console.WriteLine($"Возврат разрешен: {(ReturnAllowed ? "Да" : "Нет")}");
        if (ReturnAmount > 0)
        {
            Console.WriteLine($"Сумма возвратов: {ReturnAmount:C}");
        }
    }
}

// === ДЕМОНСТРАЦИЯ РАБОТЫ ===
class Program
{
    static void Main(string[] args)
    {
        Console.WriteLine("=== ДЕМОСТРАЦИЯ: КОНСТРУКТОРЫ + ВЗАИМОДЕЙСТВИЕ ОБЪЕКТОВ ===\n");

        // 1. Создание объектов (конструкторы используют геттеры/сеттеры)
        Console.WriteLine("1. Создание фактур:");
        var goodsInv = new GoodsInvoice("GI-001", new DateTime(2024, 1, 15), new DateTime(2024, 1, 20));
        var serviceInv = new ServiceInvoice("SI-001", new DateTime(2024, 1, 16), new DateTime(2024, 1, 25));
        var combinedInv = new CombinedInvoice("CI-001", new DateTime(2024, 1, 17), true);

        // 2. Добавление позиций (используются сеттеры для валидации)
        Console.WriteLine("\n2. Добавление позиций:");
        goodsInv.AddLine(new LineItem("Ноутбук", 2, 1500));
        goodsInv.AddLine(new LineItem("Мышь", 5, 25));
        goodsInv.AddLine(new LineItem("Мышь", 3, 30)); // Та же позиция - будет объединена!

        serviceInv.AddLine(new LineItem("Консультация", 10, 100));

        combinedInv.AddLine(new LineItem("Товар 1", 3, 200));
        combinedInv.AddLine(new LineItem("Услуга 1", 1, 500));

        // 3. Взаимодействие: передача позиции между фактурами
        Console.WriteLine("\n3. Взаимодействие - передача позиции:");
        goodsInv.TransferLineTo(combinedInv, "Мышь");

        // 4. Взаимодействие: объединение фактур
        Console.WriteLine("\n4. Взаимодействие - объединение фактур:");
        serviceInv.MergeWith(combinedInv);

        // 5. Взаимодействие: сравнение фактур
        Console.WriteLine("\n5. Взаимодействие - сравнение:");
        Console.WriteLine($"Фактура {goodsInv.InvoiceNumber} больше {combinedInv.InvoiceNumber}? {goodsInv.IsGreaterThan(combinedInv)}");

        // 6. Взаимодействие: общая сумма нескольких фактур
        Console.WriteLine("\n6. Взаимодействие - общая сумма:");
        decimal grandTotal = Invoice.GetCombinedTotal(goodsInv, serviceInv, combinedInv);
        Console.WriteLine($"Сумма всех фактур: {grandTotal:C}");

        // 7. Взаимодействие: товары между GoodsInvoice
        Console.WriteLine("\n7. Взаимодействие - объединение поставок:");
        var goodsInv2 = new GoodsInvoice("GI-002", new DateTime(2024, 1, 16), new DateTime(2024, 1, 22));
        goodsInv2.AddLine(new LineItem("Клавиатура", 10, 50));
        
        var combinedSupply = goodsInv.CreateCombinedSupply(goodsInv2, "GI-COMBINED");
        if (combinedSupply != null)
        {
            combinedSupply.DisplayInfo();
        }

        // 8. Взаимодействие: возврат в CombinedInvoice
        Console.WriteLine("\n8. Взаимодействие - возвраты:");
        var combinedInv2 = new CombinedInvoice("CI-002", new DateTime(2024, 1, 18), true);
        combinedInv2.AddLine(new LineItem("Возвратный товар", 1, 300));
        combinedInv2.AddReturn(300);
        
        combinedInv.ApplyReturnFrom(combinedInv2);

        // 9. Взаимодействие: импорт в комбинированную фактуру
        Console.WriteLine("\n9. Взаимодействие - импорт из других типов:");
        var newCombined = new CombinedInvoice("CI-NEW", DateTime.Now, true);
        newCombined.ImportFromGoodsInvoice(goodsInv);
        newCombined.ImportFromServiceInvoice(serviceInv);
        newCombined.DisplayInfo();

        // 10. Демонстрация валидации через сеттеры
        Console.WriteLine("\n10. Демонстрация валидации (сеттеры):");
        try
        {
            var invalidItem = new LineItem("", -5, -100);
        }
        catch (ArgumentException ex)
        {
            Console.WriteLine($"[VALIDATION] Ошибка: {ex.Message}");
        }

        // 11. Финальный вывод
        Console.WriteLine("\n=== ФИНАЛЬНЫЙ ОТЧЕТ ===");
        goodsInv.DisplayInfo();
        combinedInv.DisplayInfo();
        
        Console.WriteLine($"\nВсего фактур создано: 6");
        Console.WriteLine("=== ПРИНЦИПЫ ООП РЕАЛИЗОВАНЫ ===");
        Console.WriteLine("✓ Инкапсуляция: свойства с геттерами/сеттерами + валидация");
        Console.WriteLine("✓ Наследование: 3 производных класса от Invoice");
        Console.WriteLine("✓ Полиморфизм: переопределенные методы AddLine, RemoveLine, CalculateTotal");
        Console.WriteLine("✓ Взаимодействие объектов: TransferLineTo, MergeWith, CreateCombinedSupply, ImportFrom...");
    }
}